# Vintage Diagnostics (pre-Phase 5 correction)

Measures three parameters currently assumed rather than measured:

1. **Is the 2016 OOT set trustworthy?** — observed vs lower-bound default rate per vintage.
2. **Where should the train window start?** — first year each bureau feature is actually collected.
3. **What outcome horizon H?** — month-on-book by which ~90% of a vintage's defaults have resolved.

Runs on the RAW ingested frame. `build_target()` must NOT run first: it drops exactly the
censored rows this notebook needs to count.


In [ ]:
from pathlib import Path

import polars as pl

from credit_risk.data.diagnostics import (
    censoring_by_vintage,
    default_hazard_by_mob,
    feature_availability,
    first_reliable_year,
    prepayment_risk_link,
)
from credit_risk.data.ingestion import load_raw_accepted_loans

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(40)

DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")
raw = load_raw_accepted_loans(DATA_PATH)
print(raw.shape)


## 1. Censoring bias per vintage

`dr_observed` is what `build_target()` currently reports (defaults / matured).
`dr_lower_bnd` is defaults / issued — the true rate if no censored loan ever defaults.
The true vintage rate lies between them; `bias_gap` is the size of the distortion.

**Decision rule:** any vintage whose `bias_gap` exceeds ~2pp cannot be used as an
evaluation set under the current target definition.


In [ ]:
censoring = censoring_by_vintage(raw)
censoring


In [ ]:
# Export for review alongside this notebook.
censoring.write_csv("../docs/vintage_censoring.csv")


## 2. Feature availability per vintage

LendingClub expanded bureau collection in stages. A feature whose null rate collapses
from ~1.0 to ~0.0 at some year is vintage dependent: its WOE `missing` bin encodes
calendar time, not credit risk, and that bin is never hit at serving time.

Half of `SCORECARD_FEATURES` is suspected to be in this group.


In [ ]:
SUSPECT_FEATURES = [
    # kept in SCORECARD_FEATURES, suspected post-2012 bureau expansion
    "acc_open_past_24mths", "mort_acc", "mths_since_recent_bc", "avg_cur_bal",
    "bc_open_to_buy", "num_rev_tl_bal_gt_0", "mo_sin_rcnt_tl", "mths_since_recent_inq",
    # control group: expected available from 2007
    "fico_range_low", "dti", "annual_inc", "revol_util", "open_acc",
    # opposite direction: suspected to start only in 2016 (absent in train)
    "disbursement_method", "initial_list_status",
]

availability = feature_availability(raw, SUSPECT_FEATURES)
availability


In [ ]:
first_reliable_year(availability, max_null_rate=0.05)


In [ ]:
availability.write_csv("../docs/vintage_feature_availability.csv")


## 3. Default timing choosing the outcome horizon H

Run on fully matured vintages only (2012–2013). Charge-off date is not published by
LendingClub, so it is proxied as `last_pymnt_d + 5` months (LC charges off at ~121 days
delinquent).

**Two things to read here:**
- *Proxy validity:* `n_default` should peak around MOB 8–18 and decline. A flat or
  bimodal shape means the proxy is wrong and H cannot be set from it.
- *Horizon:* the MOB where `cum_share` reaches ~0.90, separately for 36 and 60 month terms.


In [ ]:
hazard = default_hazard_by_mob(raw, vintages=[2012, 2013])

for term in sorted(hazard["term_months"].unique()):
    sub = hazard.filter(pl.col("term_months") == term)
    for q in (0.50, 0.75, 0.90, 0.95):
        mob = sub.filter(pl.col("cum_share") >= q)["mob"].min()
        print(f"term={term}m  {q:.0%} of defaults resolved by MOB {mob}")
    print()


In [ ]:
hazard.filter(pl.col("term_months") == 36).head(40)


## 4. Does censoring also inflate measured AUC?

Censoring inflates the default rate unconditionally. It inflates *discrimination* only
if who-prepays correlates with risk — in that case the matured subset is a risk-widened
sample and its AUC is optimistic too.

**Decision rule:** if early payers average more than ~10 FICO points above on-schedule
payers, the reported OOT AUC of 0.721 is optimistic, not just the OOT default rate.


In [ ]:
prepay = prepayment_risk_link(raw, vintages=[2012, 2013])
prepay


In [ ]:
fico_gap = (
    prepay.pivot(on="early_payer", index="term_months", values="mean_fico")
    .with_columns((pl.col("true") - pl.col("false")).alias("fico_gap_early_minus_scheduled"))
)
fico_gap


## Summary parameters this notebook decides

| Parameter | Source | Value |
|---|---|---|
| 2016 OOT usable as-is? | section 1, `bias_gap` | |
| `train_start` (new) | section 2, `first_reliable_year` | |
| Horizon H, term 36 | section 3, `cum_share` >= 0.90 | |
| Horizon H, term 60 | section 3, `cum_share` >= 0.90 | |
| OOT AUC also inflated? | section 4, `fico_gap` | |

Fill this in, then it feeds directly into: rewriting `data/target.py` to a fixed
outcome window, shifting `split.train_end`'s lower bound in `configs/base.yaml`,
and re-running every number in `docs/modeling_findings.md`.
